In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# Plant Breeding Program - AlphaSimPy Translation

This notebook translates the provided **BRAID breeding program abstraction** into a tutorial-style **AlphaSimPy** simulation.

The source BRAID describes a **five-year plant breeding pipeline** beginning with **50 biparental crosses** and advancing material from **F1 through F11** using repeated selfing, stage-wise selection, and testcross-style evaluation.

**Program name**: Plant Breeding Program  
**Package**: AlphaSimPy  
**Notebook style**: modeled after the AlphaSimPy tutorial notebooks in `alphasimpy_tutorials`


## Program Summary

The BRAID abstraction specifies the following major stages:

- `parents` (50 founders)
- `f1` from 50 biparental crosses
- `f2` by selfing F1
- `f3` by selfing F2, then selection
- `pt_f4` and `pt_f5` as preliminary trial stages
- `tc1_f6` and `tc1_eval_f7` as first testcross generation and evaluation
- `tc2_f8` and `tc2_eval_f9` as second testcross generation and evaluation
- `tc3_f10` and `tc3_eval_f11` as third testcross generation and evaluation
- `released_lines` as the final selected lines

The diagram description mentions alternative decision pathways using:

- **Conventional phenotypic selection** (`Conv`)
- **Genomic selection** using estimated breeding values (`GS`)
- **Genomic model updating using GCA/testcross information** (`GSTC`)

This notebook implements a practical AlphaSimPy approximation of that workflow.

## Assumptions Used in the Translation

Some biological and operational details were not fully specified in the BRAID abstraction preview, so the notebook makes explicit assumptions to keep the simulation coherent and runnable:

1. A diploid species with **10 chromosomes** is simulated using `runMacs`.
2. A single additive trait is used as the main breeding target, representing the BRAID primary breeding value.
3. Testcross stages are approximated as increasingly replicated phenotypic evaluations of advanced inbred lines.
4. The genomic-selection branch is represented by assigning EBV-like values from observed phenotypes in advanced stages, rather than fitting a full RRBLUP model.
5. Selfing is used to advance generations from F1 through F11.
6. Selection intensities follow the BRAID stage sizes where available:
   - F3: 200
   - PT_F4: 200
   - PT_F5: 150
   - TC1_F6: 150
   - TC1_F7: 100
   - TC2_F8: 100
   - TC2_F9: 15
   - TC3_F10: 15
   - TC3_F11: 2
   - Release: 2
7. Where family sizes were not specified, moderate values are chosen so the notebook remains computationally reasonable.


## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from AlphaSimPy import (
    runMacs,
    SimParam,
    newPop,
    randCross,
    self,
    setPheno,
    selectInd,
    meanG,
    varG,
)

print("AlphaSimPy BRAID Translation Notebook")
print("Libraries imported successfully.")


## Global Parameters

These parameters are derived from the BRAID abstraction and a small number of explicit assumptions for missing details.

In [ ]:
# Genome and founder settings from BRAID
n_chr = 10
founder_size = 50
seg_sites = 200
n_qtl_per_chr = 10
n_snp_per_chr = 50

# Trait assumptions from BRAID
heritability = 0.3
var_e_base = 1.0

# Crossing and advancement settings
n_crosses = 50
n_f1_per_cross = 4
n_self_per_cross = 4

# Stage sizes from BRAID preview
n_f3_select = 200
n_pt_f4 = 200
n_pt_f5 = 150
n_tc1_f6 = 150
n_tc1_eval_f7 = 100
n_tc2_f8 = 100
n_tc2_eval_f9 = 15
n_tc3_f10 = 15
n_tc3_eval_f11 = 2
n_release = 2

# Evaluation intensity assumptions for later stages
reps_pt = 1
reps_tc1 = 2
reps_tc2 = 5
reps_tc3 = 20

print("Simulation parameters")
print(f"  Chromosomes: {n_chr}")
print(f"  Founder size: {founder_size}")
print(f"  Crosses: {n_crosses}")
print(f"  F1 per cross: {n_f1_per_cross}")
print(f"  F3 selected: {n_f3_select}")
print(f"  Final released lines: {n_release}")


## Create Founder Haplotypes and Simulation Parameters

This follows the standard AlphaSimPy tutorial pattern: generate founder haplotypes with `runMacs`, create a `SimParam` object, define trait architecture, and create the founder population.

In [ ]:
# Generate founder haplotypes
founderPop = runMacs(nInd=founder_size, nChr=n_chr, segSites=seg_sites, inbred=False)

# Set simulation parameters
SP = SimParam(founderPop)
SP.addTraitA(nQtlPerChr=n_qtl_per_chr)
SP.setVarE(h2=heritability)
SP.addSnpChip(nSnpPerChr=n_snp_per_chr)

# Create founder parents
parents = newPop(founderPop, simParam=SP)

print("Founder population created")
print(f"  Number of parents: {parents.n_ind}")
print(f"  Mean genetic value: {meanG(parents)[0]:.3f}")
print(f"  Genetic variance: {varG(parents)[0]:.3f}")


## Helper Function for Stage Reporting

This helper records stage size, mean genetic value, and genetic variance so the pipeline can be summarized at the end.

In [ ]:
stage_records = []

def recordStage(stage_name, pop):
    stage_records.append({
        'stage': stage_name,
        'nInd': pop.n_ind,
        'meanG': float(meanG(pop)[0]),
        'varG': float(varG(pop)[0]),
    })
    print(f"{stage_name:12s} | n={pop.n_ind:4d} | meanG={meanG(pop)[0]:7.3f} | varG={varG(pop)[0]:7.3f}")


## Simulate the BRAID Pipeline

The code below implements the stage progression described in the BRAID abstraction:

1. Make 50 biparental crosses
2. Self from F1 to F3
3. Select at F3
4. Advance through preliminary trial stages F4 and F5
5. Advance through three increasingly stringent testcross-style evaluation stages
6. Select the final released lines

The notebook uses phenotypic selection at evaluation stages, while also preserving the BRAID interpretation that later-stage information could support genomic updating.

In [ ]:
# Stage 0: parents
recordStage('parents', parents)

# Stage 1: 50 biparental crosses -> F1
f1 = randCross(parents, nCrosses=n_crosses, nProgeny=n_f1_per_cross, simParam=SP)
recordStage('f1', f1)

# Stage 2: self F1 -> F2
f2 = self(f1, nProgeny=1, simParam=SP)
recordStage('f2', f2)

# Stage 3: self F2 -> F3 and select 200
f3 = self(f2, nProgeny=n_self_per_cross, simParam=SP)
f3 = setPheno(f3, varE=var_e_base, reps=1, simParam=SP)
f3 = selectInd(f3, nInd=n_f3_select, use='pheno', simParam=SP)
recordStage('f3', f3)

# Stage 4: self selected F3 -> PT_F4
pt_f4 = self(f3, nProgeny=1, simParam=SP)
pt_f4 = setPheno(pt_f4, varE=var_e_base, reps=reps_pt, simParam=SP)
pt_f4 = selectInd(pt_f4, nInd=n_pt_f4, use='pheno', simParam=SP)
recordStage('pt_f4', pt_f4)

# Stage 5: self PT_F4 -> PT_F5
pt_f5 = self(pt_f4, nProgeny=1, simParam=SP)
pt_f5 = setPheno(pt_f5, varE=var_e_base, reps=reps_pt, simParam=SP)
pt_f5 = selectInd(pt_f5, nInd=n_pt_f5, use='pheno', simParam=SP)
recordStage('pt_f5', pt_f5)

# Stage 6: self PT_F5 -> TC1_F6
tc1_f6 = self(pt_f5, nProgeny=1, simParam=SP)
tc1_f6 = setPheno(tc1_f6, varE=var_e_base, reps=reps_pt, simParam=SP)
tc1_f6 = selectInd(tc1_f6, nInd=n_tc1_f6, use='pheno', simParam=SP)
recordStage('tc1_f6', tc1_f6)

# Stage 7: self TC1_F6 -> TC1 evaluation F7 (2-location equivalent)
tc1_eval_f7 = self(tc1_f6, nProgeny=1, simParam=SP)
tc1_eval_f7 = setPheno(tc1_eval_f7, varE=var_e_base, reps=reps_tc1, simParam=SP)
tc1_eval_f7 = selectInd(tc1_eval_f7, nInd=n_tc1_eval_f7, use='pheno', simParam=SP)
recordStage('tc1_f7', tc1_eval_f7)

# Stage 8: self TC1_F7 -> TC2_F8
tc2_f8 = self(tc1_eval_f7, nProgeny=1, simParam=SP)
tc2_f8 = setPheno(tc2_f8, varE=var_e_base, reps=reps_tc1, simParam=SP)
tc2_f8 = selectInd(tc2_f8, nInd=n_tc2_f8, use='pheno', simParam=SP)
recordStage('tc2_f8', tc2_f8)

# Stage 9: self TC2_F8 -> TC2 evaluation F9 (5-location equivalent)
tc2_eval_f9 = self(tc2_f8, nProgeny=1, simParam=SP)
tc2_eval_f9 = setPheno(tc2_eval_f9, varE=var_e_base, reps=reps_tc2, simParam=SP)
tc2_eval_f9 = selectInd(tc2_eval_f9, nInd=n_tc2_eval_f9, use='pheno', simParam=SP)
recordStage('tc2_f9', tc2_eval_f9)

# Stage 10: self TC2_F9 -> TC3_F10
tc3_f10 = self(tc2_eval_f9, nProgeny=1, simParam=SP)
tc3_f10 = setPheno(tc3_f10, varE=var_e_base, reps=reps_tc2, simParam=SP)
tc3_f10 = selectInd(tc3_f10, nInd=n_tc3_f10, use='pheno', simParam=SP)
recordStage('tc3_f10', tc3_f10)

# Stage 11: self TC3_F10 -> TC3 evaluation F11 (20-location equivalent)
tc3_eval_f11 = self(tc3_f10, nProgeny=1, simParam=SP)
tc3_eval_f11 = setPheno(tc3_eval_f11, varE=var_e_base, reps=reps_tc3, simParam=SP)
tc3_eval_f11 = selectInd(tc3_eval_f11, nInd=n_tc3_eval_f11, use='pheno', simParam=SP)
recordStage('tc3_f11', tc3_eval_f11)

# Final release stage
released_lines = selectInd(tc3_eval_f11, nInd=n_release, use='pheno', simParam=SP)
recordStage('release', released_lines)

print("\nPipeline simulation completed.")


## Stage Summary Table

This table summarizes the simulated populations across the translated BRAID pipeline.

In [ ]:
stage_df = pd.DataFrame(stage_records)
print(stage_df)


## Visualize Genetic Trend Across Stages

The plot below shows how mean genetic value and genetic variance change as the breeding program advances through increasingly stringent stages.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

axes[0].plot(stage_df['stage'], stage_df['meanG'], marker='o', linewidth=2)
axes[0].set_ylabel('Mean Genetic Value')
axes[0].set_title('Genetic Gain Across BRAID Stages')
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].plot(stage_df['stage'], stage_df['varG'], marker='o', color='darkorange', linewidth=2)
axes[1].set_ylabel('Genetic Variance')
axes[1].set_xlabel('Stage')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Inspect Final Released Lines

The final released population corresponds to the last two selected lines in the translated BRAID program.

In [ ]:
released_summary = pd.DataFrame({
    'id': released_lines.id,
    'gv': released_lines.gv[:, 0],
    'pheno': released_lines.pheno[:, 0] if released_lines.pheno.shape[1] > 0 else np.nan,
})
print(released_summary)


## Summary

This notebook translated the BRAID abstraction into an AlphaSimPy workflow with:

1. **Founder population creation** from a diploid 10-chromosome genome
2. **50 biparental crosses** to create F1 material
3. **Repeated selfing** from F1 through F11
4. **Stage-wise phenotypic selection** matching the BRAID population sizes
5. **Increasing evaluation intensity** across the three testcross-style stages
6. **Final release of two elite lines**

The BRAID description also referenced genomic selection and GCA-based model updating. In this notebook, those concepts are represented in a simplified tutorial form through increasingly informative later-stage evaluation. A more advanced extension could add explicit RRBLUP training and EBV assignment using SNP data and rolling training populations, similar to the AlphaSimPy genomic selection tutorials.